In [4]:
# Setup: imports and environment checks
import os
import sys
from pathlib import Path

# Ensure project paths
ROOT = Path(r"c:\Users\junhongs\Desktop\capstone\evaluation")
# PDF_PATH = ROOT / "material" / "UAS-OATH-Token-Implementation-Guide-6.0.1-GA (AI).pdf"

PDF_PATH = ROOT / "material" / "AccessMatrix-Supported-Platforms-6.0.1-GA (AI).pdf"

OUTPUT_DIR = ROOT / "dataset" /"uas_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDF exists: {PDF_PATH.exists()} -> {PDF_PATH}")

# Soft dependency checks
missing = []
for pkg in ["ragas", "langchain_community", "langchain_openai", "openai", "pypdf", "pandas"]:
    try:
        __import__(pkg)
    except Exception:
        missing.append(pkg)

if missing:
    print("Missing packages detected:\n - " + "\n - ".join(missing))
    print("Install them in this kernel, for example:")
    print("%pip install ragas langchain-community langchain-openai openai pypdf pandas tqdm")
else:
    print("All required packages found.")

PDF exists: True -> c:\Users\junhongs\Desktop\capstone\evaluation\material\AccessMatrix-Supported-Platforms-6.0.1-GA (AI).pdf
All required packages found.


In [ ]:
# Load and chunk the PDF into LangChain Documents
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

assert PDF_PATH.exists(), f"PDF not found at {PDF_PATH}"

loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()
print(f"Loaded {len(docs)} page-level documents")

# Chunk to ~512 tokens equivalent (~3000-4000 chars) for better synthesis
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", ".", " "]
)
chunked_docs = text_splitter.split_documents(docs)
print(f"Chunked into {len(chunked_docs)} documents")

# Show a preview
for i, d in enumerate(chunked_docs[:2]):
    print(f"Chunk {i} | {len(d.page_content)} chars | meta: {d.metadata}")

Loaded 13 page-level documents
Chunked into 20 documents
Chunk 0 | 48 chars | meta: {'producer': 'Developer Express Inc. DXperience (tm) v24.1.7', 'creator': 'Microsoft Office Word', 'creationdate': '2025-09-11T03:54:05+00:00', 'title': 'AccessMatrix-Supported-Platforms-6.0', 'author': 'ClickHelp.com', 'moddate': '2025-09-11T03:54:05+00:00', 'source': 'c:\\Users\\junhongs\\Desktop\\capstone\\evaluation\\material\\AccessMatrix-Supported-Platforms-6.0.1-GA (AI).pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}
Chunk 1 | 115 chars | meta: {'producer': 'Developer Express Inc. DXperience (tm) v24.1.7', 'creator': 'Microsoft Office Word', 'creationdate': '2025-09-11T03:54:05+00:00', 'title': 'AccessMatrix-Supported-Platforms-6.0', 'author': 'ClickHelp.com', 'moddate': '2025-09-11T03:54:05+00:00', 'source': 'c:\\Users\\junhongs\\Desktop\\capstone\\evaluation\\material\\AccessMatrix-Supported-Platforms-6.0.1-GA (AI).pdf', 'total_pages': 13, 'page': 1, 'page_label': '2'}


In [6]:
# Verify API key is visible to this kernel and optionally load from .env
import os

# Optional: auto-load from a .env file if present
try:
    from dotenv import load_dotenv  # type: ignore
    loaded = load_dotenv()
    if loaded:
        print("Loaded environment from .env")
except Exception:
    pass  # python-dotenv not installed; that's okay

api_key = os.environ.get("OPENAI_API_KEY", "")
masked = (api_key[:4] + "***" + api_key[-4:]) if api_key else None
print("OPENAI_API_KEY set:", bool(api_key), f"({masked})" if masked else "(None)")

# Tip for VS Code/Jupyter:
# If you set $env:OPENAI_API_KEY in a separate PowerShell window, you may need to
# restart the Jupyter kernel so this process sees the new environment.

Loaded environment from .env
OPENAI_API_KEY set: True (sk-s***mWMA)


In [ ]:
# ...existing code...

domain_prompt = """
### GOAL
Generate DIVERSE, CHALLENGING technical questions for evaluating a RAG system supporting i-sprint products.

### REQUIREMENTS
1. **Avoid trivial questions** like "What is X?" or "List the platforms"
2. **Focus on scenarios** engineers actually face:
   - Troubleshooting version conflicts
   - Migration planning between versions
   - Security configuration requirements
   - Integration compatibility checks

3. **Question Patterns to USE**:
   - "How would you handle..." (scenario-based)
   - "What are the implications of..." (reasoning)
   - "Compare the requirements for..." (analysis)
   - "What changes are needed when upgrading from X to Y?" (practical)

4. **FORBIDDEN Patterns**:
   - "What is...?" (too basic)
   - "List all..." (redundant with docs)
   - Direct fact lookups answerable by ctrl+F

5. **GROUNDING**: Every answer must cite exact text from source_context.

### EXAMPLES
GOOD: "When migrating from AccessMatrix 5.x to 6.0.1-GA, what database engine changes must be considered for MySQL deployments?"
BAD: "What platforms does AccessMatrix support?"
"""

In [ ]:
# Import Persona class and create persona objects
from ragas.testset.persona import Persona

personas = [
    Persona(
        name="Technical Analyst",
        role_description="Focuses on detailed system specifications and API documentation"
    ),
    Persona(
        name="Novice User",
        role_description="Asks simple questions using layman terms and basic functionality"
    ),


    Persona(
        name="DevOps Engineer",
        role_description="Troubleshoots production deployments, version upgrades, and compatibility issues. Asks scenario-based questions about migration paths and configuration conflicts."
    ),
    Persona(
        name="Security Architect", 
        role_description="Evaluates authentication protocols, authorization flows, and compliance requirements. Asks about OAuth/SAML implementations and security implications."
    )
]



In [14]:

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

# Define the model explicitly (e.g., gpt-4o for high quality generation)
openai_model = "gpt-4o-mini" 

generator_llm = LangchainLLMWrapper(ChatOpenAI(model=openai_model, temperature=0.1))  
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    persona_list=personas
)

# Define specific distribution to ensure test hardness
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.2)
] # reasoning is that abstract queries are harder to ground reliably. 

dataset = generator.generate_with_langchain_docs(
    chunked_docs, # Use chunked_docs, not docs (which are full pages)
    testset_size=3, # Start small to test
    query_distribution=query_distribution
)






C:\Users\junhongs\AppData\Local\Temp\ipykernel_41804\3610537777.py:14: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model=openai_model, temperature=0.1))
C:\Users\junhongs\AppData\Local\Temp\ipykernel_41804\3610537777.py:15: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
Applying SummaryExtractor:   0%|          | 0/13 [00:00<?, ?it/s]C:\Users\junhongs\AppData\Local\Temp\ipykernel_41804\3610537777.py:15:

In [ ]:
# Add this as a new code cell after defining domain_prompt
from datasets import Dataset

answer_prompt = """
Answer the question using only the provided document content. Avoid hallucination.
Provide a concise, accurate, and technically detailed response for a software engineer.
"""

# Convert LangChain Documents -> HF Dataset with a 'text' field
hf_docs = Dataset.from_list([{"text": d.page_content, "metadata": d.metadata} for d in chunked_docs])

# Use the customizable API (available in newer ragas versions)
try:
    dataset = generator.generate(
        documents=hf_docs,
        num_questions=1,
        question_prompt_template=domain_prompt,
        answer_prompt_template=answer_prompt
    )
except TypeError:
    # Fallback for older ragas: no custom templates available in this path
    print("generator.generate(...) with custom templates not supported in this ragas version.")
    print("Falling back to generate_with_langchain_docs without custom prompt.")
    dataset = generator.generate_with_langchain_docs(
        documents=chunked_docs,
        testset_size=10,
        prompt=domain_prompt,
    )



In [16]:
df = dataset.to_pandas()
print(df.head())

                                          user_input  \
0  What platfoms does AccessMatrix support in ver...   
1                     What is Apach Tomcat used for?   
2  What are the versions of the USOClient for And...   
3  What are the versions of USOClient available f...   
4  What updates were made regarding Kubernetes 1....   

                                  reference_contexts  \
0  [AccessMatrix Supported Platforms - 6.0.1-GA (...   
1  [1. Supported Platforms for AccessMatrix\nSupp...   
2  [<1-hop>\n\nUSOClient iOS v5.7.8 iOS\niOS 15\n...   
3  [<1-hop>\n\nUSOClient iOS v5.7.8 iOS\niOS 15\n...   
4  [<1-hop>\n\no Container Orchestration: \n§ Upd...   

                                           reference persona_name query_style  \
0  AccessMatrix supports various platforms in ver...  Novice User  MISSPELLED   
1  Apache Tomcat is used as a web application ser...  Novice User  MISSPELLED   
2  The USOClient for Android version 5.7.5 suppor...          NaN         NaN   
3 

In [17]:
# Save dataset in JSONL and CSV for cross-framework evaluations
import json
import pandas as pd
import numpy as np

# Inspect available columns
print("Columns:", list(df.columns))

def _is_nonempty_value(x):
    if x is None:
        return False
    if isinstance(x, str):
        return x.strip() != ""
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) > 0
    if isinstance(x, np.ndarray):
        return x.size > 0
    try:
        import math
        if isinstance(x, float) and math.isnan(x):
            return False
    except Exception:
        pass
    return True
    
# Try to harmonize to a simple schema: query, ground_truth, contexts(list[str])
def infer_columns(row):
    # query
    query = None
    for k in ["user_input", "question", "query", "prompt"]:
        if k in row and pd.notna(row[k]):
            query = row[k]
            break
    # ground truth answer
    gt = None
    for k in ["reference", "ground_truth", "expected_output", "answer"]:
        if k in row:
            val = row[k]
            if _is_nonempty_value(val):
                gt = val
                break
    # contexts
    contexts = None
    for k in ["contexts", "reference_contexts", "contexts_text", "documents"]:
        if k in row:
            val = row[k]
            if not _is_nonempty_value(val):
                continue
            # Ensure list[str]
            if isinstance(val, str):
                contexts = [val]
            elif isinstance(val, (list, tuple)):
                # Some entries might be dicts with 'page_content'
                first = val[0] if len(val) else None
                if isinstance(first, dict) and "page_content" in first:
                    contexts = [d.get("page_content", "") for d in val]
                else:
                    contexts = [str(v) for v in val]
            elif isinstance(val, np.ndarray):
                if val.size: 
                    first = val.flat[0]
                    if isinstance(first, dict) and "page_content" in first:
                        contexts = [d.get("page_content", "") for d in val.tolist()]
                    else:
                        contexts = [str(v) for v in val.tolist()]
            elif isinstance(val, dict) and "page_content" in val:
                contexts = [val.get("page_content", "")]            
            else:
                contexts = [str(val)]
            break
    return query, gt, contexts or []

records = []
for _, row in df.iterrows():
    q, gt, ctx = infer_columns(row)
    records.append({
        "query": q,
        "ground_truth": gt,
        "contexts": ctx,
    })

# File paths
jsonl_path = OUTPUT_DIR / "test.jsonl"
csv_path = OUTPUT_DIR / "test.csv"

# Write JSONL
with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Write CSV (flatten contexts)
pd.DataFrame({
    "query": [r["query"] for r in records],
    "ground_truth": [r["ground_truth"] for r in records],
    "contexts_joined": ["\n\n".join(r["contexts"]) for r in records]
}).to_csv(csv_path, index=False, encoding="utf-8")

print("Saved:")
print(" -", jsonl_path)
print(" -", csv_path)

Columns: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']
Saved:
 - c:\Users\junhongs\Desktop\capstone\evaluation\dataset\uas_dataset\test.jsonl
 - c:\Users\junhongs\Desktop\capstone\evaluation\dataset\uas_dataset\test.csv


In [27]:
# Convert dataset/UAS-OATH.jsonl (JSON array or JSON Lines) to CSV
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

# Expect ROOT defined earlier in the notebook
json_path = ROOT / "dataset" / "UAS-OATH.json"
csv_path = ROOT / "dataset" / "UAS-OATH.csv"

assert json_path.exists(), f"JSON/JSONL file not found: {json_path}"

text = json_path.read_text(encoding="utf-8").strip()

# Try parsing as JSON array; fallback to JSONL
try:
    if text.startswith("["):
        data = json.loads(text)
    else:
        raise ValueError("Not a JSON array; try JSONL")
except Exception:
    # JSON Lines (one JSON object per line); ignore empty lines and '//' comments
    data = [json.loads(line) for line in text.splitlines() if line.strip() and not line.strip().startswith("//")]

# Ensure list of dicts
if isinstance(data, dict):
    data = [data]
assert isinstance(data, list), "Parsed data must be a list of objects"

# DataFrame
df_json = pd.DataFrame(data)

# Order columns if present
preferred_cols = ["question_type", "question", "ground_truth_answer", "source_context"]
cols = [c for c in preferred_cols if c in df_json.columns] + [c for c in df_json.columns if c not in preferred_cols]
if cols:
    df_json = df_json[cols]

# Write CSV
df_json.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Wrote {len(df_json)} rows to: {csv_path}")

# Preview
display(df_json.head(3))

Wrote 10 rows to: c:\Users\junhongs\Desktop\capstone\evaluation\dataset\UAS-OATH.csv


,question_type,question,ground_truth_answer,source_context
0,Deployment/Configuration,What configuration setting in the Global Setti...,The 'Randomize Reservation Pool' should be set...,Randomize Reservation Pool This is to randomiz...
1,Integration/Development,When utilizing the `/token/OATH/create-assign-...,The token seed is first encrypted using RSA wi...,The application needs to generate a key pair a...
2,Troubleshooting/Error Handling,What is the physical cause mentioned in the so...,The internal clock may drift due to environmen...,OTP token internal clock may drift (for enviro...


In [31]:
# Convert dataset/UAS-OATH.jsonl (JSON array or JSON Lines) to CSV
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

# Expect ROOT defined earlier in the notebook
json_path = ROOT / "dataset" / "AM-HSM.json"
csv_path = ROOT / "dataset" / "AM-HSM.csv"

assert json_path.exists(), f"JSON/JSONL file not found: {json_path}"

text = json_path.read_text(encoding="utf-8").strip()

# Try parsing as JSON array; fallback to JSONL
try:
    if text.startswith("["):
        data = json.loads(text)
    else:
        raise ValueError("Not a JSON array; try JSONL")
except Exception:
    # JSON Lines (one JSON object per line); ignore empty lines and '//' comments
    data = [json.loads(line) for line in text.splitlines() if line.strip() and not line.strip().startswith("//")]

# Ensure list of dicts
if isinstance(data, dict):
    data = [data]
assert isinstance(data, list), "Parsed data must be a list of objects"

# DataFrame
df_json = pd.DataFrame(data)

# Order columns if present
preferred_cols = ["question_type", "question", "ground_truth_answer", "source_context"]
cols = [c for c in preferred_cols if c in df_json.columns] + [c for c in df_json.columns if c not in preferred_cols]
if cols:
    df_json = df_json[cols]

# Write CSV
df_json.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Wrote {len(df_json)} rows to: {csv_path}")

# Preview
display(df_json.head(3))

Wrote 10 rows to: c:\Users\junhongs\Desktop\capstone\evaluation\dataset\AM-HSM.csv


,question_type,question,ground_truth_answer,source_context
0,Deployment/Configuration,Edge Case — What is the unsupported deployment...,On-premise deployment of AccessMatrix that int...,AccessMatrix must be deployed in Linux VM on G...
1,Integration/Development,What authentication strategy does AccessMatrix...,AccessMatrix uses the Application Default Cred...,AccessMatrix uses Application Default Credenti...
2,Deployment/Configuration,What three specific GCP configuration paramete...,The administrator must specify the Key Ring Na...,"5. Specify the Key Ring Name (e.g. ""ISPRINT_KR..."
